In [5]:
import torch
import torch.nn as nn
from torchvision import datasets
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cuda":
    print(torch.cuda.get_device_name(0))

# Load raw MNIST once
train_raw = datasets.MNIST("./data", train=True, download=True)
test_raw = datasets.MNIST("./data", train=False, download=True)

# Convert once, normalise once, move to GPU once
X_train = train_raw.data.float().div(255.0)
X_train = (X_train - 0.1307) / 0.3081
X_train = X_train.view(-1, 28 * 28).to(device)

y_train = (train_raw.targets % 2).to(device)

X_test = test_raw.data.float().div(255.0)
X_test = (X_test - 0.1307) / 0.3081
X_test = X_test.view(-1, 28 * 28).to(device)

y_test = (test_raw.targets % 2).to(device)

model = nn.Sequential(
    nn.Linear(28 * 28, 64),
    nn.ReLU(),
    nn.Linear(64, 2),
).to(device)

opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        logits = model(X)              # shape: [N, 10]
        preds = logits.argmax(dim=1)   # predicted digit 0-9
        acc = (preds == y).float().mean()
    return acc.item()

epochs = 100
batch_size = 2048

train_losses, test_losses = [], []

for epoch in range(epochs):
    model.train()

    perm = torch.randperm(X_train.size(0), device=device)
    total_loss = 0.0

    for i in range(0, X_train.size(0), batch_size):
        idx = perm[i:i + batch_size]
        x = X_train[idx]
        y = y_train[idx]

        opt.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        opt.step()

        total_loss += loss.detach() * x.size(0)

    train_loss = (total_loss / X_train.size(0)).item()
    train_losses.append(train_loss)

    model.eval()
    with torch.no_grad():
        test_loss = loss_fn(model(X_test), y_test).item()
    test_losses.append(test_loss)

    train_acc = accuracy(model, X_train, y_train)
    test_acc = accuracy(model, X_test, y_test)

    print(f"epoch {epoch+1}: train {train_loss:.4f} test {test_loss:.4f}")

# plt.plot(train_losses, label="train")
# plt.plot(test_losses, label="test")
# plt.xlabel("epoch")
# plt.ylabel("loss")
# plt.legend()
# plt.show()


print(
        f"Total number of epochs: {epochs}; "
        f"Final train loss {train_loss:.4f}; "
        f"Final test loss {test_loss:.4f}; "
        f"Final train acc {train_acc:.4f}; "
        f"Final test acc {test_acc:.4f}."
    )

device: cpu
epoch 1: train 0.3549 test 0.2371
epoch 2: train 0.1971 test 0.1608
epoch 3: train 0.1371 test 0.1210
epoch 4: train 0.1089 test 0.0985
epoch 5: train 0.0912 test 0.0859
epoch 6: train 0.0798 test 0.0761
epoch 7: train 0.0722 test 0.0709
epoch 8: train 0.0650 test 0.0670
epoch 9: train 0.0597 test 0.0614
epoch 10: train 0.0559 test 0.0583
epoch 11: train 0.0515 test 0.0573
epoch 12: train 0.0485 test 0.0537
epoch 13: train 0.0449 test 0.0515
epoch 14: train 0.0414 test 0.0498
epoch 15: train 0.0393 test 0.0486
epoch 16: train 0.0374 test 0.0481
epoch 17: train 0.0357 test 0.0468
epoch 18: train 0.0329 test 0.0461
epoch 19: train 0.0311 test 0.0448
epoch 20: train 0.0293 test 0.0436
epoch 21: train 0.0281 test 0.0443
epoch 22: train 0.0262 test 0.0432
epoch 23: train 0.0244 test 0.0420
epoch 24: train 0.0235 test 0.0429
epoch 25: train 0.0229 test 0.0416
epoch 26: train 0.0207 test 0.0408
epoch 27: train 0.0199 test 0.0413
epoch 28: train 0.0186 test 0.0418
epoch 29: train 0